<a href="https://colab.research.google.com/github/LordSurov/123/blob/main/Lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 5. **

Для  $n$-канальной системы массового обслуживания с ограничением на длину очереди $m$ составьте дифференциальные уравнения для вероятностей нахождения в заданных состояниях в зависимости от времени. Найдите эти вероятности при определенном в соответствии с вариантом значении $t$, а также при $t\rightarrow \infty$. Канал иногда может выходить из строя. Заявка, которая обслуживается в момент отказа канала ставится в очередь, если там есть места, в противном случае она покидает систему необслуженной. Входящий поток, поток обслуживания, поток отказов и поток восстановления простейшие с соответствующими интенсивностями $\lambda, \mu, \nu, \gamma$. Количество клиентов, от которых могут поступать заявки на обслуживание $k$. Начальные условия $P_0(0)=1$.

Найти (теоретически и экспериментально):
 - вероятность простоя;
 - вероятность образования очереди;
 - абсолютную пропускную способность;
 - среднюю длину очереди;
 - среднее время нахождения заявок в системе;
 - среднее число заявок в системе.
 - среднее время нахождения в очереди.
 - оценить, что приводит к большему увеличению абсолютной пропускной способности системы - увеличение количества каналов или увеличение длины очереди при неизменности остальных характеристик?

# Параметры варианта

In [ ]:
Variant <- 14
set.seed(Variant + 643)

n <- sample(2:4, 1)          # число каналов
m <- sample(4:18, 1)         # макс. длина очереди
mu <- runif(1)
lambda <- runif(1)

if (lambda > mu) {
  tmp <- lambda
  lambda <- mu
  mu <- tmp
}

gamma <- runif(1)
nu <- runif(1)

if (gamma < nu) {
  tmp <- nu
  nu <- gamma
  gamma <- tmp
}

if (sample(0:1, 1) == 1) {
  k <- sample(4:7, 1)
} else {
  k <- Inf
}

t_target <- runif(1)

params <- data.frame(
  lambda = lambda,
  mu = mu,
  nu = nu,
  gamma = gamma,
  k = ifelse(is.infinite(k), "Inf", as.character(k)),
  n = n,
  m = m,
  t = t_target
)

print(params)

      lambda        mu       nu     gamma   k n m         t
1 0.09391977 0.2621519 0.174463 0.6805274 Inf 3 7 0.3693336


λ=0.09391977 - интенсивность входящего потока

μ=0.2621519 - интенсивность потока обслуживания

ν=0.174463 — интенсивность потока отказов каналов

γ=0.6805274 — интенсивность потока восстановления

k=∞ — бесконечный источник заявок

n=3 — число каналов

m=7 — максимальная длина очереди

t=0.3693336

# Пространство состояний

## Формирование пространства состояний

Состояние системы задаётся парой:

$$
(i, j),
$$

где:

- $i$ — число исправных каналов;
- $j$ — число заявок в системе.

Для каждого допустимого числа исправных каналов $i$ перебираются все допустимые значения числа заявок $j$.

Если число источников бесконечно, то максимальное число заявок в системе ограничивается только суммой числа исправных каналов и длины очереди:

$$
j \le i + m.
$$

Если число источников конечно, то дополнительно выполняется ограничение:

$$
j \le k.
$$

Таким образом, формируется полное множество состояний системы.

In [ ]:
states <- data.frame(i = integer(), j = integer())

for (i in 0:n) {
  if (is.infinite(k)) {
    j_max <- i + m
  } else {
    j_max <- min(k, i + m)
  }

  for (j in 0:j_max) {
    states <- rbind(states, data.frame(i = i, j = j))
  }
}

rownames(states) <- NULL
states$state_id <- seq_len(nrow(states))
states$label <- sprintf("(%d,%d)", states$i, states$j)

cat("Всего состояний:", nrow(states), "\n")
cat("i — исправных каналов, j — заявок в системе\n")
print(states)

Всего состояний: 38 
i — исправных каналов, j — заявок в системе
   i  j state_id  label
1  0  0        1  (0,0)
2  0  1        2  (0,1)
3  0  2        3  (0,2)
4  0  3        4  (0,3)
5  0  4        5  (0,4)
6  0  5        6  (0,5)
7  0  6        7  (0,6)
8  0  7        8  (0,7)
9  1  0        9  (1,0)
10 1  1       10  (1,1)
11 1  2       11  (1,2)
12 1  3       12  (1,3)
13 1  4       13  (1,4)
14 1  5       14  (1,5)
15 1  6       15  (1,6)
16 1  7       16  (1,7)
17 1  8       17  (1,8)
18 2  0       18  (2,0)
19 2  1       19  (2,1)
20 2  2       20  (2,2)
21 2  3       21  (2,3)
22 2  4       22  (2,4)
23 2  5       23  (2,5)
24 2  6       24  (2,6)
25 2  7       25  (2,7)
26 2  8       26  (2,8)
27 2  9       27  (2,9)
28 3  0       28  (3,0)
29 3  1       29  (3,1)
30 3  2       30  (3,2)
31 3  3       31  (3,3)
32 3  4       32  (3,4)
33 3  5       33  (3,5)
34 3  6       34  (3,6)
35 3  7       35  (3,7)
36 3  8       36  (3,8)
37 3  9       37  (3,9)
38 3 10       38 (3,10)

## Функция поиска номера состояния

Для удобства дальнейших расчётов каждому состоянию присваивается уникальный номер state_id.

Функция get_state_id(i, j, states_df) по заданным значениям $i$ и $j$ находит номер соответствующего состояния в таблице состояний.

Эта функция используется при построении переходов между состояниями и матрицы интенсивностей.

In [ ]:
get_state_id <- function(i, j, states_df) {
  id <- states_df$state_id[states_df$i == i & states_df$j == j]
  if (length(id) == 0) return(NA_integer_)
  as.integer(id[1])
}

## Формирование таблицы переходов

На этом этапе для каждого состояния перечисляются все возможные переходы в другие состояния.

Рассматриваются следующие события:

1. **Поступление заявки**  
   Если в системе есть место, то число заявок увеличивается на единицу.

2. **Завершение обслуживания**  
   Если в системе есть обслуживаемые заявки, то число заявок уменьшается на единицу.

3. **Отказ простаивающего канала**  
   Если среди исправных каналов есть свободные, то при отказе уменьшается число исправных каналов, но число заявок в системе не меняется.

4. **Отказ занятого канала**  
   Если отказал занятый канал, то возможны два случая:
   - если очередь не заполнена, заявка остаётся в системе;
   - если очередь заполнена, заявка теряется.

5. **Восстановление канала**  
   При восстановлении число исправных каналов увеличивается на единицу.

Для каждого перехода записываются:
- исходное состояние;
- конечное состояние;
- интенсивность перехода;
- тип события.

In [ ]:
transitions <- data.frame(
  from = integer(),
  to = integer(),
  rate = numeric(),
  type = character(),
  stringsAsFactors = FALSE
)

for (idx in seq_len(nrow(states))) {
  i <- states$i[idx]
  j <- states$j[idx]
  from_id <- states$state_id[idx]

  busy <- min(i, j)          # занятые каналы
  idle_ch <- max(0, i - j)   # простаивающие каналы

  # 1. Поступление заявки
  if (is.infinite(k)) {
    if (j < i + m) {
      to_id <- get_state_id(i, j + 1, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id,
          to = to_id,
          rate = lambda,
          type = "arrival"
        ))
      }
    }
  } else {
    if (j < k && j < i + m) {
      to_id <- get_state_id(i, j + 1, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id,
          to = to_id,
          rate = (k - j) * lambda,
          type = "arrival"
        ))
      }
    }
  }

  # 2. Завершение обслуживания
  if (j > 0 && busy > 0) {
    to_id <- get_state_id(i, j - 1, states)
    if (!is.na(to_id)) {
      transitions <- rbind(transitions, data.frame(
        from = from_id,
        to = to_id,
        rate = busy * mu,
        type = "service"
      ))
    }
  }

  # 3. Отказ простаивающего канала
  if (i > 0 && idle_ch > 0) {
    to_id <- get_state_id(i - 1, j, states)
    if (!is.na(to_id)) {
      transitions <- rbind(transitions, data.frame(
        from = from_id,
        to = to_id,
        rate = idle_ch * nu,
        type = "fail_idle"
      ))
    }
  }

  # 4. Отказ занятого канала
  if (i > 0 && busy > 0) {
    if (j < i + m) {
      to_id <- get_state_id(i - 1, j, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id,
          to = to_id,
          rate = busy * nu,
          type = "fail_busy_keep"
        ))
      }
    } else {
      to_id <- get_state_id(i - 1, j - 1, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id,
          to = to_id,
          rate = busy * nu,
          type = "fail_busy_loss"
        ))
      }
    }
  }

  # 5. Восстановление канала
  if (i < n) {
    to_id <- get_state_id(i + 1, j, states)
    if (!is.na(to_id)) {
      transitions <- rbind(transitions, data.frame(
        from = from_id,
        to = to_id,
        rate = (n - i) * gamma,
        type = "repair"
      ))
    }
  }
}

cat("Число переходов:", nrow(transitions), "\n")
head(transitions, 20)

Число переходов: 121 


,from,to,rate,type
,<int>,<int>,<dbl>,<chr>
1,1,2,0.09391977,arrival
2,1,9,2.04158218,repair
3,2,3,0.09391977,arrival
4,2,10,2.04158218,repair
5,3,4,0.09391977,arrival
6,3,11,2.04158218,repair
7,4,5,0.09391977,arrival
8,4,12,2.04158218,repair
9,5,6,0.09391977,arrival


## Построение матрицы интенсивностей переходов

По таблице переходов строится матрица интенсивностей $Q$ непрерывного марковского процесса.

Элемент матрицы $Q_{ab}$ показывает интенсивность перехода из состояния $a$ в состояние $b$.

Диагональные элементы матрицы определяются по правилу:

$$
Q_{aa} = - \sum_{b \ne a} Q_{ab}.
$$

Это необходимо для того, чтобы сумма элементов каждой строки матрицы была равна нулю, что является обязательным свойством матрицы генератора марковского процесса.

После построения матрицы дополнительно проверяется:
- отсутствие значений `NA`, `NaN`, `Inf`;
- близость сумм строк к нулю.

In [ ]:
N <- nrow(states)
Q <- matrix(0, nrow = N, ncol = N)

for (tt in seq_len(nrow(transitions))) {
  fr <- transitions$from[tt]
  to <- transitions$to[tt]
  rt <- transitions$rate[tt]
  Q[fr, to] <- Q[fr, to] + rt
}

for (r in seq_len(N)) {
  Q[r, r] <- -sum(Q[r, -r])
}

cat("Проверка сумм строк Q:\n")
print(range(rowSums(Q)))

cat("Есть ли NA:", any(is.na(Q)), "\n")
cat("Есть ли NaN:", any(is.nan(Q)), "\n")
cat("Есть ли Inf:", any(is.infinite(Q)), "\n")

Проверка сумм строк Q:
[1] 0 0
Есть ли NA: FALSE 
Есть ли NaN: FALSE 
Есть ли Inf: FALSE 


In [ ]:
if (!require(expm)) {
  install.packages("expm", repos = "https://cloud.r-project.org")
  library(expm)
} else {
  library(expm)
}

## Нахождение вероятностей состояний в момент времени $t$

В начальный момент времени система считается пустой, а все каналы исправными.  
Следовательно, начальное состояние имеет вид:

$$
(i, j) = (n, 0).
$$

Начальный вектор вероятностей содержит единицу в позиции, соответствующей начальному состоянию, и нули во всех остальных позициях.

Вероятности состояний в момент времени $t$ находятся по формуле:

$$
P(t) = P(0)e^{Qt}.
$$

In [ ]:
initial_state <- get_state_id(n, 0, states)   # в начале все каналы исправны, заявок нет
y0 <- rep(0, N)
y0[initial_state] <- 1

P_t <- as.vector(y0 %*% expm(Q * t_target))
names(P_t) <- states$label

cat("Нестационарное решение для t =", t_target, ":\n")
print(round(P_t, 8))

Нестационарное решение для t = 0.3693336 :
     (0,0)      (0,1)      (0,2)      (0,3)      (0,4)      (0,5)      (0,6) 
0.00016309 0.00000549 0.00000009 0.00000000 0.00000000 0.00000000 0.00000000 
     (0,7)      (1,0)      (1,1)      (1,2)      (1,3)      (1,4)      (1,5) 
0.00000000 0.00837120 0.00027669 0.00000466 0.00000005 0.00000000 0.00000000 
     (1,6)      (1,7)      (1,8)      (2,0)      (2,1)      (2,2)      (2,3) 
0.00000000 0.00000000 0.00000000 0.14313603 0.00473227 0.00007822 0.00000088 
     (2,4)      (2,5)      (2,6)      (2,7)      (2,8)      (2,9)      (3,0) 
0.00000001 0.00000000 0.00000000 0.00000000 0.00000000 0.00000000 0.81580875 
     (3,1)      (3,2)      (3,3)      (3,4)      (3,5)      (3,6)      (3,7) 
0.02697176 0.00044586 0.00000491 0.00000004 0.00000000 0.00000000 0.00000000 
     (3,8)      (3,9)     (3,10) 
0.00000000 0.00000000 0.00000000 


## Нахождение стационарных вероятностей

Стационарные вероятности состояний находятся из системы линейных уравнений:

$$
\pi Q = 0,
$$

с дополнительным условием нормировки:

$$
\sum \pi_i = 1.
$$

Полученный вектор $\pi$ показывает вероятности состояний в установившемся режиме работы системы.

In [ ]:
A <- t(Q)
A[N, ] <- 1
b <- rep(0, N)
b[N] <- 1

P_inf <- solve(A, b)
names(P_inf) <- states$label

cat("Стационарное решение:\n")
print(round(P_inf, 8))
cat("Сумма вероятностей =", sum(P_inf), "\n")

Стационарное решение:
     (0,0)      (0,1)      (0,2)      (0,3)      (0,4)      (0,5)      (0,6) 
0.00559387 0.00224467 0.00053406 0.00010233 0.00001778 0.00000295 0.00000048 
     (0,7)      (1,0)      (1,1)      (1,2)      (1,3)      (1,4)      (1,5) 
0.00000009 0.06847141 0.02446437 0.00532868 0.00096502 0.00016252 0.00002656 
     (1,6)      (1,7)      (1,8)      (2,0)      (2,1)      (2,2)      (2,3) 
0.00000429 0.00000070 0.00000010 0.26864198 0.09705820 0.01845288 0.00307639 
     (2,4)      (2,5)      (2,6)      (2,7)      (2,8)      (2,9)      (3,0) 
0.00049733 0.00007974 0.00001276 0.00000204 0.00000031 0.00000003 0.34992689 
     (3,1)      (3,2)      (3,3)      (3,4)      (3,5)      (3,6)      (3,7) 
0.12662382 0.02373592 0.00336579 0.00051108 0.00007994 0.00001266 0.00000201 
     (3,8)      (3,9)     (3,10) 
0.00000030 0.00000004 0.00000000 
Сумма вероятностей = 1 


## Представление вероятностей состояний в табличной форме

Для удобства анализа результаты представляются в виде таблиц:

- вероятности состояний в момент времени $t$;
- стационарные вероятности состояний.

Это позволяет увидеть, какие состояния являются наиболее вероятными в начальный и установившийся моменты работы системы.

In [ ]:
res_t <- cbind(states[, c("i", "j")], P_t = round(P_t, 8))
res_inf <- cbind(states[, c("i", "j")], P_inf = round(P_inf, 8))

cat("Вероятности состояний в момент t:\n")
print(res_t)

cat("\nСтационарные вероятности:\n")
print(res_inf)

Вероятности состояний в момент t:
       i  j        P_t
(0,0)  0  0 0.00016309
(0,1)  0  1 0.00000549
(0,2)  0  2 0.00000009
(0,3)  0  3 0.00000000
(0,4)  0  4 0.00000000
(0,5)  0  5 0.00000000
(0,6)  0  6 0.00000000
(0,7)  0  7 0.00000000
(1,0)  1  0 0.00837120
(1,1)  1  1 0.00027669
(1,2)  1  2 0.00000466
(1,3)  1  3 0.00000005
(1,4)  1  4 0.00000000
(1,5)  1  5 0.00000000
(1,6)  1  6 0.00000000
(1,7)  1  7 0.00000000
(1,8)  1  8 0.00000000
(2,0)  2  0 0.14313603
(2,1)  2  1 0.00473227
(2,2)  2  2 0.00007822
(2,3)  2  3 0.00000088
(2,4)  2  4 0.00000001
(2,5)  2  5 0.00000000
(2,6)  2  6 0.00000000
(2,7)  2  7 0.00000000
(2,8)  2  8 0.00000000
(2,9)  2  9 0.00000000
(3,0)  3  0 0.81580875
(3,1)  3  1 0.02697176
(3,2)  3  2 0.00044586
(3,3)  3  3 0.00000491
(3,4)  3  4 0.00000004
(3,5)  3  5 0.00000000
(3,6)  3  6 0.00000000
(3,7)  3  7 0.00000000
(3,8)  3  8 0.00000000
(3,9)  3  9 0.00000000
(3,10) 3 10 0.00000000

Стационарные вероятности:
       i  j      P_inf
(0,0)  0  0 0.00559

## Функция вычисления характеристик системы

На основе вероятностей состояний вычисляются основные характеристики системы:

- вероятность простоя;
- вероятность образования очереди;
- абсолютная пропускная способность;
- средняя длина очереди;
- среднее число заявок в системе;
- среднее время нахождения заявки в системе;
- среднее время нахождения заявки в очереди.

Среднее время нахождения заявок определяется по формулам Литтла:

$$
W = \frac{L}{A},
$$

$$
W_q = \frac{L_q}{A},
$$

где:

- $L$ — среднее число заявок в системе;
- $L_q$ — средняя длина очереди;
- $A$ — абсолютная пропускная способность.

In [ ]:
calc_metrics <- function(prob, states_df, lambda, mu, n, m, k) {
  idle_prob <- 0
  queue_prob <- 0
  avg_queue <- 0
  avg_system <- 0
  abs_throughput <- 0
  block_prob <- 0

  for (idx in seq_len(nrow(states_df))) {
    i <- states_df$i[idx]
    j <- states_df$j[idx]
    p <- prob[idx]

    busy <- min(i, j)
    q_len <- max(0, j - i)
    full_system <- (j == i + m)

    if (j == 0) {
      idle_prob <- idle_prob + p
    }

    if (q_len > 0) {
      queue_prob <- queue_prob + p
    }

    avg_queue <- avg_queue + q_len * p
    avg_system <- avg_system + j * p
    abs_throughput <- abs_throughput + busy * mu * p

    if (full_system) {
      block_prob <- block_prob + p
    }
  }

  if (abs_throughput > 0) {
    avg_time_system <- avg_system / abs_throughput
    avg_time_queue <- avg_queue / abs_throughput
  } else {
    avg_time_system <- NA
    avg_time_queue <- NA
  }

  list(
    idle_prob = idle_prob,
    queue_prob = queue_prob,
    absolute_throughput = abs_throughput,
    avg_queue = avg_queue,
    avg_system = avg_system,
    avg_time_system = avg_time_system,
    avg_time_queue = avg_time_queue,
    block_prob = block_prob
  )
}

## Теоретические характеристики системы

С помощью найденных вероятностей состояний рассчитываются характеристики:

1. **Для момента времени $t$** — нестационарный режим;
2. **Для стационарного режима** — при $t \to \infty$.

Это позволяет сравнить поведение системы в начале работы и после выхода на установившийся режим.

In [ ]:
metrics_t <- calc_metrics(P_t, states, lambda, mu, n, m, k)
metrics_inf <- calc_metrics(P_inf, states, lambda, mu, n, m, k)

print_metrics <- function(metrics, title) {
  cat("\n========================================\n")
  cat(title, "\n")
  cat("========================================\n")

  cat(sprintf("Вероятность простоя системы: %.6f\n", metrics$idle_prob))
  cat(sprintf("Вероятность образования очереди: %.6f\n", metrics$queue_prob))
  cat(sprintf("Абсолютная пропускная способность: %.6f\n", metrics$absolute_throughput))
  cat(sprintf("Средняя длина очереди: %.6f\n", metrics$avg_queue))
  cat(sprintf("Среднее число заявок в системе: %.6f\n", metrics$avg_system))
  cat(sprintf("Среднее время пребывания в системе: %.6f\n", metrics$avg_time_system))
  cat(sprintf("Среднее время пребывания в очереди: %.6f\n", metrics$avg_time_queue))
  cat(sprintf("Вероятность потери заявки: %.6f\n", metrics$block_prob))
}

print_metrics(metrics_t, "Характеристики системы в момент времени t")
print_metrics(metrics_inf, "Стационарные характеристики системы")


Характеристики системы в момент времени t 
Вероятность простоя системы: 0.967479
Вероятность образования очереди: 0.000011
Абсолютная пропускная способность: 0.008664
Средняя длина очереди: 0.000011
Среднее число заявок в системе: 0.033062
Среднее время пребывания в системе: 3.815897
Среднее время пребывания в очереди: 0.001314
Вероятность потери заявки: 0.000000

Стационарные характеристики системы 
Вероятность простоя системы: 0.692634
Вероятность образования очереди: 0.013665
Абсолютная пропускная способность: 0.093920
Средняя длина очереди: 0.016680
Среднее число заявок в системе: 0.374945
Среднее время пребывания в системе: 3.992185
Среднее время пребывания в очереди: 0.177602
Вероятность потери заявки: 0.000000


## Имитационная модель системы

Для экспериментальной проверки результатов строится имитационная модель системы.

Моделирование выполняется как дискретно-событийный процесс, в котором в случайные моменты времени могут происходить следующие события:

- поступление заявки;
- завершение обслуживания;
- отказ свободного канала;
- отказ занятого канала;
- восстановление канала.

В процессе моделирования накапливаются временные характеристики системы, по которым затем вычисляются экспериментальные оценки исследуемых показателей.

In [ ]:
simulate_system <- function(Tmax, lambda, mu, nu, gamma, n, m, k, seed = 123) {
  set.seed(seed)

  time <- 0
  i <- n
  j <- 0

  area_idle <- 0
  area_queue <- 0
  area_q <- 0
  area_j <- 0
  completed <- 0

  while (time < Tmax) {
    busy <- min(i, j)
    idle_ch <- max(0, i - j)
    q_len <- max(0, j - i)

    if (is.infinite(k)) {
      rate_arrival <- if (j < i + m) lambda else 0
    } else {
      rate_arrival <- if (j < k && j < i + m) (k - j) * lambda else 0
    }

    rate_service <- busy * mu
    rate_fail_idle <- idle_ch * nu
    rate_fail_busy <- busy * nu
    rate_repair <- (n - i) * gamma

    total_rate <- rate_arrival + rate_service + rate_fail_idle + rate_fail_busy + rate_repair
    if (total_rate <= 0) break

    dt <- rexp(1, rate = total_rate)
    if (time + dt > Tmax) dt <- Tmax - time

    area_idle <- area_idle + dt * as.numeric(j == 0)
    area_queue <- area_queue + dt * as.numeric(q_len > 0)
    area_q <- area_q + dt * q_len
    area_j <- area_j + dt * j

    time <- time + dt
    if (time >= Tmax) break

    u <- runif(1) * total_rate

    if (u < rate_arrival) {
      j <- j + 1

    } else if (u < rate_arrival + rate_service) {
      j <- j - 1
      completed <- completed + 1

    } else if (u < rate_arrival + rate_service + rate_fail_idle) {
      i <- i - 1

    } else if (u < rate_arrival + rate_service + rate_fail_idle + rate_fail_busy) {
      if (j < i + m) {
        i <- i - 1
      } else {
        i <- i - 1
        j <- j - 1
      }

    } else {
      i <- i + 1
    }
  }

  abs_throughput <- completed / Tmax

  list(
    idle_prob = area_idle / Tmax,
    queue_prob = area_queue / Tmax,
    absolute_throughput = abs_throughput,
    avg_queue = area_q / Tmax,
    avg_system = area_j / Tmax,
    avg_time_system = ifelse(abs_throughput > 0, (area_j / Tmax) / abs_throughput, NA),
    avg_time_queue = ifelse(abs_throughput > 0, (area_q / Tmax) / abs_throughput, NA)
  )
}

## Экспериментальные оценки характеристик

По результатам имитационного моделирования вычисляются экспериментальные значения основных характеристик системы:

- вероятность простоя;
- вероятность образования очереди;
- абсолютная пропускная способность;
- средняя длина очереди;
- среднее число заявок в системе;
- среднее время пребывания в системе;
- среднее время пребывания в очереди.

Эти значения используются для сравнения с теоретическими результатами.

In [ ]:
sim_metrics <- simulate_system(
  Tmax = 100000,
  lambda = lambda,
  mu = mu,
  nu = nu,
  gamma = gamma,
  n = n,
  m = m,
  k = k,
  seed = 123
)

cat("Экспериментальные оценки:\n")
print(sim_metrics)

Экспериментальные оценки:
$idle_prob
[1] 0.6970912

$queue_prob
[1] 0.01392357

$absolute_throughput
[1] 0.09293

$avg_queue
[1] 0.01705735

$avg_system
[1] 0.3717185

$avg_time_system
[1] 3.999984

$avg_time_queue
[1] 0.1835506



## Сравнение теоретических и экспериментальных результатов

На данном этапе строится сравнительная таблица, в которой для каждой характеристики системы указываются:

- теоретическое значение;
- экспериментальное значение.

Если модель построена корректно, то соответствующие значения должны быть близки друг к другу.

In [ ]:
comparison <- data.frame(
  metric = c(
    "idle_prob",
    "queue_prob",
    "absolute_throughput",
    "avg_queue",
    "avg_system",
    "avg_time_system",
    "avg_time_queue"
  ),
  theoretical = c(
    metrics_inf$idle_prob,
    metrics_inf$queue_prob,
    metrics_inf$absolute_throughput,
    metrics_inf$avg_queue,
    metrics_inf$avg_system,
    metrics_inf$avg_time_system,
    metrics_inf$avg_time_queue
  ),
  experimental = c(
    sim_metrics$idle_prob,
    sim_metrics$queue_prob,
    sim_metrics$absolute_throughput,
    sim_metrics$avg_queue,
    sim_metrics$avg_system,
    sim_metrics$avg_time_system,
    sim_metrics$avg_time_queue
  )
)

print(comparison)

               metric theoretical experimental
1           idle_prob  0.69263416   0.69709122
2          queue_prob  0.01366485   0.01392357
3 absolute_throughput  0.09391971   0.09293000
4           avg_queue  0.01668035   0.01705735
5          avg_system  0.37494489   0.37171848
6     avg_time_system  3.99218516   3.99998366
7      avg_time_queue  0.17760226   0.18355055


## Функция для анализа абсолютной пропускной способности

Для ответа на последний вопрос лабораторной работы требуется определить, что сильнее увеличивает абсолютную пропускную способность системы:

- увеличение числа каналов $n$;
- увеличение длины очереди $m$.

Для этого создаётся отдельная функция, которая для заданных параметров системы строит пространство состояний, матрицу переходов, находит стационарное распределение и вычисляет абсолютную пропускную способность.

In [ ]:
build_system_and_get_throughput <- function(lambda, mu, nu, gamma, n, m, k) {
  states <- data.frame(i = integer(), j = integer())

  for (i in 0:n) {
    if (is.infinite(k)) {
      j_max <- i + m
    } else {
      j_max <- min(k, i + m)
    }
    for (j in 0:j_max) {
      states <- rbind(states, data.frame(i = i, j = j))
    }
  }

  rownames(states) <- NULL
  states$state_id <- seq_len(nrow(states))

  get_state_id <- function(i, j, states_df) {
    id <- states_df$state_id[states_df$i == i & states_df$j == j]
    if (length(id) == 0) return(NA_integer_)
    as.integer(id[1])
  }

  transitions <- data.frame(
    from = integer(),
    to = integer(),
    rate = numeric(),
    type = character(),
    stringsAsFactors = FALSE
  )

  for (idx in seq_len(nrow(states))) {
    i <- states$i[idx]
    j <- states$j[idx]
    from_id <- states$state_id[idx]

    busy <- min(i, j)
    idle_ch <- max(0, i - j)

    if (is.infinite(k)) {
      if (j < i + m) {
        to_id <- get_state_id(i, j + 1, states)
        if (!is.na(to_id)) {
          transitions <- rbind(transitions, data.frame(
            from = from_id, to = to_id, rate = lambda, type = "arrival"
          ))
        }
      }
    } else {
      if (j < k && j < i + m) {
        to_id <- get_state_id(i, j + 1, states)
        if (!is.na(to_id)) {
          transitions <- rbind(transitions, data.frame(
            from = from_id, to = to_id, rate = (k - j) * lambda, type = "arrival"
          ))
        }
      }
    }

    if (j > 0 && busy > 0) {
      to_id <- get_state_id(i, j - 1, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id, to = to_id, rate = busy * mu, type = "service"
        ))
      }
    }

    if (i > 0 && idle_ch > 0) {
      to_id <- get_state_id(i - 1, j, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id, to = to_id, rate = idle_ch * nu, type = "fail_idle"
        ))
      }
    }

    if (i > 0 && busy > 0) {
      if (j < i + m) {
        to_id <- get_state_id(i - 1, j, states)
        if (!is.na(to_id)) {
          transitions <- rbind(transitions, data.frame(
            from = from_id, to = to_id, rate = busy * nu, type = "fail_busy_keep"
          ))
        }
      } else {
        to_id <- get_state_id(i - 1, j - 1, states)
        if (!is.na(to_id)) {
          transitions <- rbind(transitions, data.frame(
            from = from_id, to = to_id, rate = busy * nu, type = "fail_busy_loss"
          ))
        }
      }
    }

    if (i < n) {
      to_id <- get_state_id(i + 1, j, states)
      if (!is.na(to_id)) {
        transitions <- rbind(transitions, data.frame(
          from = from_id, to = to_id, rate = (n - i) * gamma, type = "repair"
        ))
      }
    }
  }

  N <- nrow(states)
  Q <- matrix(0, N, N)

  for (tt in seq_len(nrow(transitions))) {
    fr <- transitions$from[tt]
    to <- transitions$to[tt]
    rt <- transitions$rate[tt]
    Q[fr, to] <- Q[fr, to] + rt
  }

  for (r in seq_len(N)) {
    Q[r, r] <- -sum(Q[r, -r])
  }

  A <- t(Q)
  A[N, ] <- 1
  b <- rep(0, N)
  b[N] <- 1

  P_inf <- solve(A, b)

  abs_throughput <- 0
  for (idx in seq_len(nrow(states))) {
    i <- states$i[idx]
    j <- states$j[idx]
    busy <- min(i, j)
    abs_throughput <- abs_throughput + busy * mu * P_inf[idx]
  }

  abs_throughput
}

In [ ]:
A_base <- build_system_and_get_throughput(lambda, mu, nu, gamma, n, m, k)
A_n_plus <- build_system_and_get_throughput(lambda, mu, nu, gamma, n + 1, m, k)
A_m_plus <- build_system_and_get_throughput(lambda, mu, nu, gamma, n, m + 1, k)

cat("Сравнение абсолютной пропускной способности:\n")
cat("Базовая система: ", A_base, "\n")
cat("При увеличении n на 1: ", A_n_plus, "\n")
cat("При увеличении m на 1: ", A_m_plus, "\n")
cat("Прирост при n+1: ", A_n_plus - A_base, "\n")
cat("Прирост при m+1: ", A_m_plus - A_base, "\n")

Сравнение абсолютной пропускной способности:
Базовая система:  0.09391971 
При увеличении n на 1:  0.09391976 
При увеличении m на 1:  0.09391976 
Прирост при n+1:  4.81033e-08 
Прирост при m+1:  4.372318e-08 


In [ ]:
if ((A_n_plus - A_base) > (A_m_plus - A_base)) {
  cat("Большее увеличение абсолютной пропускной способности даёт увеличение числа каналов.\n")
} else if ((A_n_plus - A_base) < (A_m_plus - A_base)) {
  cat("Большее увеличение абсолютной пропускной способности даёт увеличение длины очереди.\n")
} else {
  cat("Оба способа дают одинаковый прирост абсолютной пропускной способности.\n")
}

Большее увеличение абсолютной пропускной способности даёт увеличение числа каналов.
